# 15 — Orquestrador `executar_pipeline`

Desenvolve o `principal` que liga a esteira. **F10, NF5.**

In [3]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
import pandas as pd
from app import dal, nucleo
from app.mercado import RendaFixa, RendaVariavel
from app.agente import Investidor

## Desenvolvimento

A função abaixo foi escrita aqui e, após os testes, movida para `app/principal.py`.

In [4]:
# Fator de desconto **anual** padrão.

BETA_ANUAL_PADRAO = 0.96


def _n_scenarios_padrao(periodos_por_ano: int) -> int:
    """Cenários de Monte Carlo adequados à frequência dos dados.
    """

    return 200_000 if periodos_por_ano <= 12 else 4_000_000


def executar_pipeline(config: dict) -> dict:
    """Executa a esteira de ponta a ponta e devolve os resultados. (NF5)

    Chaves de ``config``:
      Frequência — **obrigatória**, sem default (ver nota):
        ``periodos_por_ano`` : 12 (mensal) ou 252 (diário).
      Dados (uma das duas):
        ``retornos`` : DataFrame com ``data`` + colunas de ativos; OU
        ``db_path`` (+ ``tabela``, default ``'retornos'``) : lê do SQLite.
      Mercado:
        ``ativos`` : colunas de risco (default: todas menos ``data`` e ``rf_col``);
        ``rf_col`` : coluna da taxa livre de risco nos dados (default ``'cdi'``);
        ``cdi_anual`` : CDI **anual**, convertido para o período por ``RendaFixa``
        com ``periodos_por_ano``. Se ausente, usa a média de ``rf_col``, que o
        ``dal`` já grava por período.
      Agente: ``gamma`` (5.0), ``w0`` (1.0);
        ``horizonte`` : T em **períodos** — obrigatório, sem default (ver nota);
        desconto: ``beta_anual`` (0.96, convertido para o período) **ou**
        ``beta`` (já por período) — passar os dois é erro.
      Simulação: ``n_scenarios`` (200k mensal / 4M diário), ``n_paths`` (5000),
        ``seed`` (42).

    Os retornos são sempre Normais e a carteira α* é sempre irrestrita (short e
    alavancagem permitidos, sem teto), fiel ao PDF (§3.1).

    Returns
    -------
    dict com ``alpha_star`` (carteira ótima), ``theta``/``consumo_inicial``,
    ``phi_hat``, ``A_t`` (Etapa 3) e ``valor_V`` (Etapa 7 — V_t na riqueza
    inicial, F11), calibração (``mu_hat``, ``sigma_hat``, ``rf``) e o resumo da
    simulação: ``E_W_T``, percentis de W_T e as trajetórias
    ``trajetoria_W_media``/``_mediana``/``_p5``/``_p95`` e ``trajetoria_c_media``.
    Devolve também a unidade de tempo efetivamente usada — ``periodos_por_ano``
    e ``beta`` (já por período) — para que quem exibe os números não precise
    refazer a conversão.
    """
    cfg = dict(config)
    coluna_data = cfg.get("coluna_data", "data")
    rf_col = cfg.get("rf_col", "cdi")

    # ── 0. Frequência — a unidade de tempo de todo o resto ──────────────────
    if "periodos_por_ano" not in cfg:
        raise ValueError(
            "config precisa de 'periodos_por_ano' (12 mensal, 252 diário). "
            "Não há default: é ele que dá unidade de tempo a 'cdi_anual', "
            "'beta_anual' e ao default de 'n_scenarios'."
        )
    ppa = int(cfg["periodos_por_ano"])
    if ppa <= 0:
        raise ValueError(f"'periodos_por_ano' deve ser positivo; veio {ppa!r}.")

    # ── 1. DAL — obter retornos ─────────────────────────────────────────────
    if cfg.get("retornos") is not None:
        retornos = cfg["retornos"]
    elif "db_path" in cfg:
        retornos = dal.ler_sqlite(cfg["db_path"], cfg.get("tabela", "retornos"))
    else:
        raise ValueError("config precisa de 'retornos' (DataFrame) ou 'db_path'.")

    colunas = [c for c in retornos.columns if c != coluna_data]
    ativos = cfg.get("ativos") or [c for c in colunas if c != rf_col]
    if not ativos:
        raise ValueError("nenhum ativo de risco identificado em 'retornos'.")

    # ── 2. Mercado — calibração (Etapa 0) ───────────────────────────────────
    if "cdi_anual" in cfg:
        rf = RendaFixa(cfg["cdi_anual"], ppa).retorno_livre_risco()
    elif rf_col in retornos.columns:
        rf = float(retornos[rf_col].mean())
    else:
        rf = float(cfg.get("rf", 0.0))
    mercado = RendaVariavel(retornos[[coluna_data] + ativos], coluna_data=coluna_data)

    # ── 3. Agente — política ótima (Etapas 1–4) ─────────────────────────────
    if "horizonte" not in cfg:
        raise ValueError(
            "config precisa de 'horizonte' (T, em períodos). Não há default: "
            "o número de períodos depende da frequência dos dados — 60 é 5 anos "
            "no mensal e ~3 meses no diário."
        )
    if "beta" in cfg and "beta_anual" in cfg:
        raise ValueError("use 'beta' (por período) OU 'beta_anual', não os dois.")
    beta = (float(cfg["beta"]) if "beta" in cfg
            else float(cfg.get("beta_anual", BETA_ANUAL_PADRAO)) ** (1.0 / ppa))
    inv = Investidor(cfg.get("gamma", 5.0), beta,
                     cfg.get("w0", 1.0), cfg["horizonte"])
    seed = cfg.get("seed", 42)
    n_scenarios = cfg.get("n_scenarios") or _n_scenarios_padrao(ppa)
    alpha = inv.carteira_otima(mercado, rf, n_scenarios=n_scenarios, seed=seed)
    theta = inv.fracoes_consumo()

    # ── 4. Simulação forward (Etapas 5–6) ───────────────────────────────────
    T, N = inv.horizonte, len(ativos)
    n_paths = cfg.get("n_paths", 5_000)
    r_paths = mercado.amostrar(n_paths * T, seed=seed + 1)
    R_paths = np.maximum(1.0 + r_paths.reshape(n_paths, T, N), 0.0)
    sim = nucleo.propagar_riqueza(inv.w0, theta, alpha, R_paths, 1.0 + rf)

    # ── 5. Resultado ────────────────────────────────────────────────────────
    W_T = sim["W"][:, -1]
    A_t = inv.coeficientes_A
    valor_V = nucleo.funcao_valor(A_t, inv.w0, inv.gamma)
    W_p5, W_p50, W_p95 = np.percentile(sim["W"], [5, 50, 95], axis=0)
    return {
        "ativos": ativos,
        "periodos_por_ano": ppa,
        "rf": rf,
        "beta": beta,
        "mu_hat": mercado.media(),
        "sigma_hat": mercado.covariancia(),
        "alpha_star": alpha,                              # carteira ótima
        "phi_hat": inv.phi_hat,
        "theta": theta,                                   # frações de consumo
        "consumo_inicial": float(theta[0] * inv.w0),      # c_0 = θ_0·W_0
        "horizonte": T,
        "E_W_T": float(W_T.mean()),
        "W_T_p5": float(np.percentile(W_T, 5)),
        "W_T_p95": float(np.percentile(W_T, 95)),
        "A_t": A_t,
        "valor_V": valor_V,
        "trajetoria_W_media": sim["W"].mean(axis=0),
        "trajetoria_W_mediana": W_p50,
        "trajetoria_W_p5": W_p5,
        "trajetoria_W_p95": W_p95,
        "trajetoria_c_media": sim["c"].mean(axis=0),
    }


**Teste** — roda a esteira e devolve resultado coerente.

In [5]:
import pandas as pd
rng = np.random.default_rng(7); ruido = rng.normal(0,0.06,200); ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=200,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido,'cdi':np.full(200,0.008)})
res = executar_pipeline({'retornos':ret,'ativos':['ibov'],'periodos_por_ano':12,
                        'cdi_anual':0.10,'gamma':5.0,'beta':0.96,'w0':1.0,'horizonte':12,'n_scenarios':40_000,'n_paths':2_000,'seed':1})
print('chaves:', list(res.keys())); print('alpha*:', res['alpha_star'], '| E[W_T]:', res['E_W_T'])

chaves: ['ativos', 'periodos_por_ano', 'rf', 'beta', 'mu_hat', 'sigma_hat', 'alpha_star', 'phi_hat', 'theta', 'consumo_inicial', 'horizonte', 'E_W_T', 'W_T_p5', 'W_T_p95', 'A_t', 'valor_V', 'trajetoria_W_media', 'trajetoria_W_mediana', 'trajetoria_W_p5', 'trajetoria_W_p95', 'trajetoria_c_media']
alpha*: [0.48558069] | E[W_T]: 0.08019876896545776


In [6]:
assert res['alpha_star'].shape==(1,) and np.isclose(res['theta'][-1],1.0) and res['E_W_T']>0
print('pipeline: PASSOU')

pipeline: PASSOU


**Teste** — a unidade de tempo não pode ser adivinhada: `periodos_por_ano` é obrigatório e é ele que converte `cdi_anual` e `beta_anual`.

In [7]:
# 1) sem 'periodos_por_ano' a esteira para, em vez de herdar 12 em silencio
base = {'retornos': ret, 'ativos': ['ibov'], 'horizonte': 12, 'n_scenarios': 20_000,
        'n_paths': 500, 'seed': 1}
try:
    executar_pipeline(base); raise SystemExit('deveria ter falhado')
except ValueError as e:
    assert 'periodos_por_ano' in str(e)

In [8]:
# 2) o mesmo cdi_anual gera R_f diferente em cada frequencia (e nao o mensal nas duas)
mensal = executar_pipeline({**base, 'periodos_por_ano': 12,  'cdi_anual': 0.1312})
diario = executar_pipeline({**base, 'periodos_por_ano': 252, 'cdi_anual': 0.1312})
assert np.isclose(mensal['rf'], 1.1312 ** (1 / 12) - 1)
assert np.isclose(diario['rf'], 1.1312 ** (1 / 252) - 1)

In [9]:
# 3) beta_anual vira beta do periodo; 'beta' cru continua aceito para quem ja converteu
assert np.isclose(diario['beta'], 0.96 ** (1 / 252))          # default anual
assert np.isclose(executar_pipeline({**base, 'periodos_por_ano': 252,
                                     'beta': 0.5})['beta'], 0.5)
try:
    executar_pipeline({**base, 'periodos_por_ano': 12, 'beta': 0.9, 'beta_anual': 0.9})
    raise SystemExit('deveria ter falhado')
except ValueError as e:
    assert 'beta_anual' in str(e)

print('R_f mensal:', mensal['rf'], '| R_f diario:', diario['rf'])
print('unidade de tempo: PASSOU')

R_f mensal: 0.010326202364327575 | R_f diario: 0.0004893221241111245
unidade de tempo: PASSOU
